In [0]:
# Step 1: Specify the path to the input CSV file
# File Name should be always be Update.csv irrepective of the name of the table getting updated.
file_path = "/mnt/EDH/02curated/CSV_to_Parquet/CSV_File/"

In [0]:
# Step 2: Specify the output path for the Parquet file
parquet_output_path = "/mnt/EDH/02curated/CSV_to_Parquet/Parquet_Data/"

In [0]:
# Step 3: Read the CSV file into a Spark DataFrame
# - format("csv"): Specifies that the input file is in CSV format
# - option("header", "true"): Indicates that the first row contains column headers
df = spark.read.format("csv").option("header", "true").load(file_path+"Update.csv")

In [0]:
# Step 4: Perform any necessary data transformations or cleaning (optional)
# For example, you could cast columns, filter rows, or handle missing values here
# df = df.withColumn("column_name", df["column_name"].cast("desired_type"))

In [0]:
from datetime import datetime

# Step 5: Write the DataFrame to Parquet format in a date-partitioned folder with subfolders for multiple runs
current_date = datetime.now().strftime("%Y-%m-%d")
base_dated_output_path = f"{parquet_output_path}{current_date}/"

def get_next_parquet_subfolder(base_path):
    i = 1
    while True:
        subfolder = f"{base_path}{i}/"
        try:
            if not any(f.name == "" for f in dbutils.fs.ls(subfolder) if f.path.endswith('/')):
                dbutils.fs.mkdirs(subfolder)
                return subfolder
        except Exception:
            dbutils.fs.mkdirs(subfolder)
            return subfolder
        i += 1

final_parquet_path = get_next_parquet_subfolder(base_dated_output_path)
df.write.mode("overwrite").parquet(final_parquet_path)

In [0]:
# Step 6: (Optional) Verify the Parquet output by reading it back and displaying the data
# result_df = spark.read.parquet(parquet_output_path)
# display(result_df)

In [0]:
# Step 7: Archive the original CSV file after conversion
from datetime import datetime

archive_base_path = file_path + "archive/"
current_date = datetime.now().strftime("%Y-%m-%d")
archive_date_path = f"{archive_base_path}{current_date}/"

def get_next_subfolder(base_path, filename):
    i = 1
    while True:
        subfolder = f"{base_path}{i}/"
        dbutils.fs.mkdirs(subfolder)
        file_exists = any(f.name == filename for f in dbutils.fs.ls(subfolder))
        if not file_exists:
            return subfolder
        i += 1

final_archive_path = get_next_subfolder(archive_date_path, "Update.csv")
dbutils.fs.mv(file_path + "Update.csv", final_archive_path + "Update.csv")